In [ ]:
# ==============================================================================# 🚀 DO NOT MODIFY: Standardized Notebook Setup# ==============================================================================# This cell is designed to work in both Google Colab and local environments.# It ensures that the environment is correctly configured by cloning (or# locating) the project repository and installing the necessary dependencies.## ------------------------------------------------------------------------------##  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):##  This cell will automatically find the repository root and configure your#  environment. Just make sure you have run: pip install -e .[dev]## ------------------------------------------------------------------------------import osimport subprocessimport sysfrom pathlib import Path# --- Configuration ---REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned# --- End of Configuration ---def find_repo_root(start_path: Path) -> Path | None:    """    Find the repository root by looking for pyproject.toml.    Searches upward from start_path until it finds pyproject.toml or hits root.    Args:        start_path: Directory to start searching from.    Returns:        Path to repository root, or None if not found.    """    current = start_path.resolve()    while current \!= current.parent:  # Stop at filesystem root        if (current / "pyproject.toml").exists():            return current        current = current.parent    return Nonedef detect_active_branch(repo_dir: Path) -> str:    """    Determine the active git branch for pulling updates.    Tries multiple methods to detect the current branch name.    Args:        repo_dir: Path to the git repository.    Returns:        Branch name (defaults to 'master' if detection fails).    """    commands = [        "git symbolic-ref --short HEAD",        "git rev-parse --abbrev-ref HEAD",    ]    for cmd in commands:        result = subprocess.run(            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True        )        if result.returncode == 0:            branch = result.stdout.strip()            if branch and not branch.startswith("origin/"):                return branch    return "master"def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:    """    Run a shell command and raise an error if it fails.    Args:        cmd: The command to run.        cwd: Optional working directory for the command.    Raises:        RuntimeError: If the command returns a non-zero exit code.    """    result = subprocess.run(cmd, shell=True, cwd=cwd)    if result.returncode \!= 0:        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")# --- Detect environment ---try:    import google.colab  # noqa: F401    IN_COLAB = Trueexcept ImportError:    IN_COLAB = False# --- Main setup logic ---if IN_COLAB:    print("☁️  Running in Google Colab. Setting up the environment...")    # Determine repository path    start_dir = Path.cwd()    if start_dir.name == REPO_DIR.name:        repo_path = start_dir    else:        repo_path = start_dir / REPO_DIR    # Clone or update repository    if not repo_path.exists():        print(f"📥 Cloning repository from {REPO_URL}...")        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")        print(f"✅ Repository cloned to {repo_path}")    else:        print(f"📂 Repository already exists at {repo_path}")        active_branch = detect_active_branch(repo_path)        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)        print(f"✅ Repository updated")    # Verify repository structure    if not (repo_path / "pyproject.toml").exists():        raise FileNotFoundError(            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "            "The repository may be corrupted."        )    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    # Install dependencies (smart installation - only installs missing packages)    from core.notebook.setup import smart_install_dependencies    result = smart_install_dependencies(        repo_path=repo_path,        include_dev=True,        verbose=True,    )    # Fail loudly if critical packages failed to install    if result["failed"]:        print(f"⚠️  WARNING: {len(result['failed'])} packages failed to install:")        for pkg in result["failed"]:            print(f"  - {pkg}")        print("You may encounter import errors. Please check your internet connection.")    print("" + "=" * 70)    print("✅ Environment setup complete\! You can now proceed with the notebook.")    print("=" * 70)else:    print("💻 Running in local environment. Configuring...")    # Find the repository root    repo_path = find_repo_root(Path.cwd())    if repo_path is None:        raise FileNotFoundError(            "Could not find repository root (no pyproject.toml found). "            "Please ensure you are running this notebook from within the "            "ADH-LLM-Tutorials-2025 repository directory."        )    print(f"✅ Found repository root: {repo_path}")    # Change working directory and update Python path    print(f"📁 Changing working directory to {repo_path}")    os.chdir(repo_path)    if str(repo_path) not in sys.path:        sys.path.insert(0, str(repo_path))    print("" + "=" * 70)    print("✅ Local environment configured successfully\!")    print("=" * 70)    print("⚠️  Please ensure you have run: pip install -e .[dev]")    print("   (Required for local development)")

# 04 - Sepsis Prediction with Transformer

## Attention is All You Need: The Transformer Architecture

In this notebook, we explore the **Transformer** architecture for sepsis prediction. Originally introduced in the landmark paper "Attention is All You Need" (Vaswani et al., 2017), Transformers have revolutionized natural language processing and are now making significant impacts in medical time-series analysis.

### What Makes Transformers Different?

Unlike RNNs and LSTMs which process sequences sequentially, Transformers use **self-attention mechanisms** to process all timesteps in parallel:

1. **Self-Attention**: Each timestep can directly attend to all other timesteps in the sequence, allowing the model to capture complex temporal relationships without the sequential bottleneck of recurrent architectures.

2. **Positional Encoding**: Since Transformers don't inherently process sequences in order, we add positional encodings to the input to inject information about the temporal ordering of measurements.

3. **Parallel Processing**: The entire sequence can be processed at once, making Transformers more computationally efficient than recurrent models on modern hardware.

For medical time-series, this architecture allows the model to directly identify relationships between distant events (e.g., connecting an early vital sign abnormality to a later clinical deterioration) without the information bottleneck inherent in recurrent architectures.

In [ ]:
# Import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

from core.config import TrainConfig, TransformerConfig
from core.data import create_dataloaders
from core.data.physionet_sepsis import get_sepsis_data
from core.models import TransformerModel
from core.notebook import ensure_project_root
from core.train import Trainer

## Step 1: Load Configuration

In [ ]:
project_root = ensure_project_root()
# Load configuration from YAML
config_path = Path("configs/transformer.yaml")
config_dict = yaml.safe_load(config_path.read_text())

# Parse into Pydantic models
model_config = TransformerConfig(**config_dict["model"])
train_config = TrainConfig(**config_dict["training"])

print("Model Configuration:")
print(model_config)
print("\nTraining Configuration:")
print(train_config)

### Understanding the Hyperparameters

Below is a summary of the hyperparameters we'll use to train this Transformer model. These parameters control both the model architecture and the training process.

**🔍 Notice the Transformer-specific parameters:**
- **`nhead: 2`** - Number of attention heads (unique to Transformer)
- **`dim_feedforward: 256`** - Size of feedforward network (unique to Transformer)
- **`learning_rate: 0.0005`** - Lower than RNN/LSTM due to different optimization dynamics
- **`batch_size: 64`** - Larger batches help Transformers learn better attention patterns
- **`epochs: 15`** - Middle ground between GRU (20) and LSTM (10)

In [ ]:
from core.notebook import display_hyperparameter_table

# Display hyperparameters in a formatted table
display_hyperparameter_table(model_config, train_config, model_type="Transformer")

### 🧪 Experimentation Guide

Want to experiment with different hyperparameters? Here's how!

#### **1. Safe Parameters to Modify (Great for Learning)**

**`learning_rate`** - Controls how quickly the model learns
- Transformers often need lower learning rates than RNNs
- Current: 0.0005 (half of GRU/LSTM's 0.001)
- **Try:** 0.0001, 0.0005 (current), 0.001, 0.002

**`dropout`** - Prevents overfitting by randomly dropping connections during training
- Transformers can be prone to overfitting due to their capacity
- **Try:** 0.1, 0.2 (current), 0.3, 0.4

**`epochs`** - Number of complete passes through the training data
- Transformers typically need moderate epochs (not too few, not too many)
- **Try:** 10, 15 (current), 20, 25

#### **2. Transformer-Specific Architectural Parameters**

**`nhead`** - Number of attention heads (⚠️ **Special constraint!**)
- **MUST divide `input_size` evenly!**
- Current: 2 heads (34 features ÷ 2 = 17 features per head)
- **Valid values for 34 features:** 1, 2 (current), 17, 34
- **Try:** 1 (single-head attention), 2 (current), 17 (fine-grained)

**`dim_feedforward`** - Size of the feedforward network in each encoder layer
- Larger values → more capacity, but slower training
- **Try:** 128, 256 (current), 512, 1024

**`num_encoder_layers`** - Number of stacked Transformer encoder layers
- More layers → deeper representation, but harder to train
- **Try:** 1, 2 (current), 3, 4

**`max_seq_length`** - Maximum sequence length supported
- Current: 5000 (much larger than needed - longest patient stay is ~1000 hours)
- **Try:** 1000, 2000, 5000 (current)
- **Note:** Larger values use more memory for positional encodings

#### **3. Comparing Transformer to RNN/LSTM**

| Parameter | GRU | LSTM | Transformer | Why Different? |
|-----------|-----|------|-------------|----------------|
| `hidden_size` | 64 | 128 | N/A | Transformer uses `input_size` directly |
| `nhead` | N/A | N/A | 2 | Unique to Transformer (multi-head attention) |
| `dim_feedforward` | N/A | N/A | 256 | Unique to Transformer (FFN layer) |
| `learning_rate` | 0.001 | 0.001 | 0.0005 | Transformers often need lower LR |
| `batch_size` | 32 | 64 | 64 | Larger batches help attention learn patterns |

#### **4. Parameters to Leave Alone**

**`input_size`** (34) - Must match the number of features in the dataset ❌  
**`train_val_split`** (0.8) - Keep consistent for fair model comparison ❌  
**`split_seed`** (42) - Keep at 42 for reproducibility ❌

#### **How to Modify Hyperparameters**

1. **Edit the YAML file:** Open `configs/transformer.yaml` in a text editor
2. **Change the values:** Modify the parameters you want to experiment with
3. **IMPORTANT for `nhead`:** Ensure it divides 34 evenly (1, 2, 17, or 34)
4. **Reload this notebook:** Restart the kernel and run all cells again
5. **Compare results:** Compare with GRU and LSTM results

#### **Expected Experimental Outcomes**

| Modification | Expected Effect |
|--------------|-----------------|
| `nhead` → 1 | Single-head attention, simpler model, may lose multi-faceted patterns |
| `nhead` → 17 | Fine-grained attention (2 features/head), more complex, may overfit |
| `dim_feedforward` → 512 | Slower training, potentially better capacity for complex patterns |
| `num_encoder_layers` → 3 | Deeper model, may capture more complex relationships, harder to train |
| `learning_rate` → 0.001 | Training may become unstable (too high for Transformer) |

#### **Understanding Attention Heads**

With **`nhead: 2`** and **`input_size: 34`**, each attention head processes **17 features**:

- **Head 1** might focus on vital signs (HR, BP, O2Sat, etc.)
- **Head 2** might focus on lab values (Lactate, BUN, Creatinine, etc.)

This parallel processing allows the model to learn different aspects of the patient state simultaneously!

#### **Reflection Questions**

Before you start training, think about:
- **Why might Transformer outperform GRU/LSTM for sepsis prediction?**  
  *Hint: Think about direct connections between distant timepoints*
  
- **What trade-off does `nhead=17` vs `nhead=2` represent?**  
  *Hint: Fine-grained vs. coarse-grained feature grouping*

- **Why is the learning rate lower (0.0005) than GRU/LSTM (0.001)?**  
  *Hint: Think about how attention mechanisms affect optimization*

Let's proceed with training!

## Step 2: Load and Prepare Data

We load the cached PhysioNet dataframe and delegate batching to the `core.data` helper, which caches per-patient tensors and performs a deterministic split.

In [ ]:
# Load the preprocessed PhysioNet dataset
sepsis_df = get_sepsis_data()

num_patients = sepsis_df["patient_id"].nunique()
print(f"Total ICU patient stays: {num_patients:,}")
print(f"Total hourly measurements: {len(sepsis_df):,}")

In [ ]:
# Build deterministic train/validation DataLoaders
train_loader, val_loader = create_dataloaders(
    train_config=train_config,
    df=sepsis_df,
)

print(
    "Train patients:",
    len(train_loader.dataset),
    "| Val patients:",
    len(val_loader.dataset),
)
print(
    f"Using split ratio {train_config.train_val_split:.0%} "
    f"with seed {train_config.split_seed}"
)

## Step 3: Initialize Model and Trainer

In [ ]:
# Create model
model = TransformerModel(model_config)
print(model)

# Define path to save the best model
model_save_path = Path("models/transformer_best.pt")

# Create trainer with checkpoint saving
trainer = Trainer(
    model, train_loader, val_loader, train_config, save_path=model_save_path
)

### ⚠️ Pre-Training Checkpoint

Before we start training, let's review our configuration one more time. This gives you a final chance to verify everything is set correctly.

In [ ]:
from core.notebook import print_pre_training_summary

# Display pre-training summary
print_pre_training_summary(model, train_loader, val_loader, train_config)

## Step 4: Run Training

In [ ]:
# Train the model and save the best checkpoint
history, best_model_path = trainer.fit()

print(f"\n✅ Training complete! Best model saved to: {best_model_path}")

## Step 5: Visualize Results

In [ ]:
from core.notebook import plot_training_dashboard

# Display comprehensive training dashboard
plot_training_dashboard(
    history, model_name="Transformer", save_path=str(best_model_path)
)